# 02 — H&M retrieval (v2: cold-start fix + đánh giá 1+99)

Notebook này huấn luyện User Tower trên lịch sử mua hàng + embedding text/ảnh (Jina CLIP v2) để gợi ý sản phẩm tiếp theo.

**Thay đổi so với v1 (đọc `docs/logs` nếu cần bối cảnh):**

| # | Vấn đề ở v1 | Sửa ở v2 |
|---|---|---|
| 1 | ~30% sự kiện test là item *chưa từng xuất hiện* trong train (cold). Hàng ID của chúng không bao giờ nhận gradient dương nên giữ nguyên giá trị khởi tạo ngẫu nhiên và được cộng vào vector item → nhiễu. | **ID-dropout** khi train (ép nhánh text/ảnh tự mang đủ tín hiệu) + **cold-fix** lúc suy luận (tắt phần ID của item có degree < ngưỡng). Ngưỡng chọn trên cửa sổ chọn checkpoint. |
| 2 | Tower chỉ học đến hết cửa sổ train, nên item mới ra trong 2 tuần cuối (selection + valid) là cold khi test. | **Refit**: sau khi chọn số epoch, train lại từ đầu trên train + selection + valid với đúng số epoch đó rồi mới chấm test (valid vẫn chấm bằng model chưa thấy valid). |
| 3 | Chỉ có full-ranking. | Thêm **sampled 1+99** (giống `sampled_ranking_evaluate` của Amazon): uniform và popularity-weighted, tách overall / warm / cold, kèm baseline random + popularity và CI 95% (cluster theo user). |
| 4 | Không có baseline "bán chạy tuần gần nhất" — baseline mạnh nhất của H&M. | Thêm `recent_popularity` làm mốc so sánh. |
| 5 | Negative sampling bằng `np.random.choice(p=...)` trên CPU (chậm). | `torch.multinomial` trên GPU, kết quả cùng phân phối. |

Tất cả có thể tắt bằng biến môi trường để tái lập v1: `HM_ID_DROPOUT=0 HM_COLD_FIX=0 HM_REFIT=0`.

## 1. Chuẩn bị môi trường

In [4]:
from datetime import timedelta
from pathlib import Path
import json
import os
import random
import time

import numpy as np
import polars as pl
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [5]:
PROFILE = os.getenv("HM_PROFILE", "quick").lower()
assert PROFILE in {"quick", "full"}
MODALITY = os.getenv("HM_MODALITY", "multimodal").lower()
assert MODALITY in {"id", "text", "image", "multimodal"}

# Kaggle Input chỉ có quyền đọc; chỉ lưu kết quả vào /kaggle/working.
DATA_DIR = Path(os.getenv("HM_DATA_DIR", os.getenv("HM_WORK_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled")))
OUTPUT_DIR = Path(os.getenv("HM_OUTPUT_DIR", "/kaggle/working/hm_retrieval"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- cold-start / refit / eval switches (tắt hết = tái lập v1) ---
ID_DROPOUT = float(os.getenv("HM_ID_DROPOUT", "0.3"))        # xác suất bỏ phần ID của MỖI item ở mỗi bước train
COLD_FIX = os.getenv("HM_COLD_FIX", "1") == "1"              # suy luận: tắt ID của item có degree < ngưỡng
COLD_GRID = [int(x) for x in os.getenv("HM_COLD_GRID", "1,2,3,5,10").split(",")]
REFIT = os.getenv("HM_REFIT", "1") == "1"                    # train lại trên train+selection+valid để chấm test
SAMPLED_NEG = int(os.getenv("HM_SAMPLED_NEG", "99"))         # 1 + 99 như Amazon
# Luật phục vụ (chỉ áp lúc suy luận, chọn trên cửa sổ selection): chỉ xếp hạng item đang bán + cộng điểm cho item cold.
TUNE_SERVING = os.getenv("HM_TUNE_SERVING", "1") == "1"
ACTIVE_GRID = [int(x) for x in os.getenv("HM_ACTIVE_GRID", "0,14,28").split(",")]             # 0 = không lọc
BONUS_GRID = [float(x) for x in os.getenv("HM_BONUS_GRID", "0,0.05,0.1,0.2").split(",")]      # cộng điểm item cold
POP_GRID = [float(x) for x in os.getenv("HM_POP_GRID", "0,0.02,0.05,0.1,0.2,0.3,0.5").split(",")]     # trọng số log(1 + số lần bán 7 ngày)
assert 0 <= ID_DROPOUT < 1

cfg = json.loads((DATA_DIR / "run_config.json").read_text())
SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Profile: {PROFILE} | Modality: {MODALITY} | Device: {device}")
print(f"ID-dropout={ID_DROPOUT} | cold-fix={COLD_FIX} {COLD_GRID} | refit={REFIT} | sampled 1+{SAMPLED_NEG}")
print(f"Read from: {DATA_DIR}\nSave to: {OUTPUT_DIR}")

Profile: quick | Modality: multimodal | Device: cuda
ID-dropout=0.3 | cold-fix=True [1, 2, 3, 5, 10] | refit=True | sampled 1+99
Read from: /kaggle/input/datasets/hoho0111/hm-dataset-filled
Save to: /kaggle/working/hm_retrieval


## 2. Đọc dữ liệu và embedding

Thêm một vector toàn 0 ở đầu để dùng làm padding. Các cửa sổ thời gian (đều 7 ngày): `train` (< 02/09) → `rerank_train` (chọn checkpoint) → `valid` → `test`.

In [6]:
def find_artifact(name):
    for path in (DATA_DIR / name, DATA_DIR / "embedding" / "hm" / name, DATA_DIR / "embedding" / name):
        if path.is_file():
            return path
    raise FileNotFoundError(f"Không thấy {name} trong {DATA_DIR} (đã thử ., embedding/hm, embedding)")

items = pl.read_parquet(DATA_DIR / "items.parquet")
item_ids = items["item_id"].to_list()
N_ITEMS = len(item_ids)

text_raw = np.load(find_artifact("text_embeddings.npy")).astype("float32")
image_raw = np.load(find_artifact("image_embeddings.npy")).astype("float32")
assert len(text_raw) == len(image_raw) == N_ITEMS, (text_raw.shape, image_raw.shape, N_ITEMS)

text = np.vstack([np.zeros((1, text_raw.shape[1]), dtype="float32"), text_raw])
image = np.vstack([np.zeros((1, image_raw.shape[1]), dtype="float32"), image_raw])

print(f"Số sản phẩm: {N_ITEMS:,}")
print(f"Text shape: {text.shape} | Image shape: {image.shape}")

Số sản phẩm: 105,542
Text shape: (105543, 512) | Image shape: (105543, 512)


In [7]:
vocab = {item_id: idx + 1 for idx, item_id in enumerate(item_ids)}

def load_events(name):
    data = pl.read_parquet(DATA_DIR / f"{name}.parquet")
    data = data.with_columns(pl.col("article_id").replace_strict(vocab, default=0).alias("idx"))
    return data.filter(pl.col("idx") > 0)

def daily_baskets(events):
    """user -> [basket_ngày_1, basket_ngày_2, ...] (mỗi basket là các idx duy nhất, đã sort)."""
    daily = events.group_by(["customer_id", "t_dat"]).agg(pl.col("idx").unique().sort().alias("basket")).sort(["customer_id", "t_dat"])
    baskets = {}
    for user, basket in zip(daily["customer_id"].to_list(), daily["basket"].to_list()):
        baskets.setdefault(user, []).append(basket)
    return baskets

def flatten_baskets(baskets):
    return [item for basket in baskets for item in basket]

def merge_baskets(*parts):
    """Ghép các cửa sổ liên tiếp theo thời gian (parts phải được truyền theo thứ tự thời gian)."""
    merged = {}
    for part in parts:
        for user, days in part.items():
            merged.setdefault(user, []).extend(days)
    return merged

def count_vector(events, last_days=None):
    """Số lần mua của từng item; last_days=7 -> chỉ tuần cuối của cửa sổ."""
    if last_days:
        events = events.filter(pl.col("t_dat") >= events["t_dat"].max() - timedelta(days=last_days - 1))
    return np.bincount(events["idx"].to_numpy(), minlength=N_ITEMS + 1).astype("float64")

ev = {name: load_events(name) for name in ("train", "rerank_train", "valid", "test")}
train_baskets = daily_baskets(ev["train"])
# Tên file cũ là rerank_train, nhưng ở notebook này nó chỉ là cửa sổ chọn checkpoint.
selection_baskets = daily_baskets(ev["rerank_train"])
valid_baskets = daily_baskets(ev["valid"])
test_baskets = daily_baskets(ev["test"])

train = {u: flatten_baskets(b) for u, b in train_baskets.items()}
selection_window = {u: flatten_baskets(b) for u, b in selection_baskets.items()}
valid = {u: flatten_baskets(b) for u, b in valid_baskets.items()}
test = {u: flatten_baskets(b) for u, b in test_baskets.items()}

# Thống kê bán hàng theo (item, ngày): ưu tiên bảng đếm của TOÀN BỘ khách (notebook 00 xuất item_daily_counts.parquet), nếu không có thì dùng mẫu.
# Mọi thống kê chỉ lấy ngày < cutoff của cửa sổ đích nên không rò rỉ nhãn.
GLOBAL_STATS = Path(os.getenv("HM_GLOBAL_STATS", str(DATA_DIR / "item_daily_counts.parquet")))
if GLOBAL_STATS.is_file():
    counts = pl.read_parquet(GLOBAL_STATS).with_columns(pl.col("article_id").replace_strict(vocab, default=0).alias("idx")).filter(pl.col("idx") > 0)
    st_idx, st_day, st_n = counts["idx"].to_numpy(), counts["t_dat"].cast(pl.Int32).to_numpy(), counts["n"].to_numpy().astype("float64")
    STATS_SOURCE = f"global ({int(st_n.sum()):,} giao dịch)"
else:
    sample_ev = pl.concat([ev[n].select("idx", "t_dat") for n in ("train", "rerank_train", "valid", "test")])
    st_idx, st_day, st_n = sample_ev["idx"].to_numpy(), sample_ev["t_dat"].cast(pl.Int32).to_numpy(), np.ones(len(sample_ev))
    STATS_SOURCE = f"mẫu 50k khách ({len(st_idx):,} giao dịch)"
LAST_DAY = int(ev["test"]["t_dat"].cast(pl.Int32).max())
CUTOFF = {"selection": LAST_DAY - 20, "valid": LAST_DAY - 13, "test": LAST_DAY - 6}   # 3 cửa sổ 7 ngày liên tiếp
print("thống kê bán hàng từ:", STATS_SOURCE)

def window_counts(window, days):
    """Số lần bán của từng item trong `days` ngày ngay trước cutoff của cửa sổ."""
    keep = (st_day < CUTOFF[window]) & (st_day >= CUTOFF[window] - days)
    return np.bincount(st_idx[keep], weights=st_n[keep], minlength=N_ITEMS + 1)

# Tuần bán chạy ngay trước cửa sổ cần dự đoán (baseline mạnh của H&M, không rò rỉ).
recent_pop = {w: window_counts(w, 7) for w in CUTOFF}

print(f"Train users: {len(train):,} | Valid users: {len(valid):,} | Test users: {len(test):,}")
print(f"Train days: {sum(map(len, train_baskets.values())):,} | Train events: {sum(map(len, train.values())):,}")

thống kê bán hàng từ: global (31,788,324 giao dịch)
Train users: 50,000 | Valid users: 3,030 | Test users: 2,799
Train days: 360,555 | Train events: 1,120,945


## 3. Mô hình

* `Tower.item_table(id_drop)`: vector item = ID (residual, nhân `sigmoid(id_gate)`) + text + ảnh (có cổng), rồi L2-normalize.
* **ID-dropout**: khi train, mỗi bước bỏ phần ID của ngẫu nhiên `id_drop` số item → nhánh nội dung phải tự phân biệt được item, nên item cold (chỉ có nội dung) nằm cùng không gian với item warm.
* **Cold-fix**: lúc suy luận, buffer `id_keep=0` cho item có degree (số lần mua trong tập fit) < ngưỡng → item chưa được học không còn bị cộng nhiễu ID khởi tạo.

In [8]:
MAXLEN = 30
D_MODEL = 128
N_NEG = int(os.getenv("HM_NUM_NEGATIVES", 128))
TEMPERATURE = 0.07

class PairData(Dataset):
    def __init__(self, daily_histories):
        self.rows = []
        for days in daily_histories.values():
            # Người chỉ có một ngày mua vẫn đóng góp tín hiệu co-purchase.
            if len(days) == 1 and len(days[0]) > 1:
                basket = days[0]
                self.rows.extend(([item for item in basket if item != target], target) for target in basket)
                continue
            history = []
            for basket in days:
                # Không học thứ tự giả giữa các món được mua cùng một ngày.
                if history:
                    self.rows.extend((history.copy(), target) for target in basket)
                history.extend(basket)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        history, target = self.rows[index]
        history = ([0] * MAXLEN + history[-MAXLEN:])[-MAXLEN:]
        return torch.tensor(history), torch.tensor(target)

class Tower(nn.Module):
    def __init__(self):
        super().__init__()
        self.id = nn.Embedding(N_ITEMS + 1, D_MODEL, padding_idx=0)
        self.text_proj = nn.Linear(text.shape[1], D_MODEL, bias=False)
        self.image_proj = nn.Linear(image.shape[1], D_MODEL, bias=False)
        self.image_gate = nn.Parameter(torch.tensor(0.0))
        # ID là residual nhỏ lúc đầu, để không lấn át embedding Jina đã pretrained.
        self.id_gate = nn.Parameter(torch.tensor(-2.0))
        self.register_buffer("text", torch.tensor(text))
        self.register_buffer("image", torch.tensor(image))
        self.register_buffer("id_keep", torch.ones(N_ITEMS + 1, 1))  # 1 = dùng ID, 0 = chỉ dùng nội dung
        self.register_buffer("degree", torch.zeros(N_ITEMS + 1))     # số lần mua trong tập fit

    def set_degree(self, degree, min_degree=0):
        """Ghi degree của tập fit và tắt ID của item có degree < min_degree (0 = không tắt gì)."""
        self.degree.copy_(torch.as_tensor(degree, dtype=torch.float32))
        keep = torch.ones_like(self.degree) if min_degree <= 0 else (self.degree >= min_degree).float()
        keep[0] = 1.0
        self.id_keep.copy_(keep[:, None])

    def item_table(self, id_drop=0.0):
        id_vectors = torch.sigmoid(self.id_gate) * self.id.weight
        if id_drop > 0:
            id_vectors = id_vectors * (torch.rand(id_vectors.shape[0], 1, device=id_vectors.device) >= id_drop).float()
        else:
            id_vectors = id_vectors * self.id_keep
        text_vectors = self.text_proj(self.text)
        image_vectors = self.image_proj(self.image)
        if MODALITY == "id":
            vectors = id_vectors
        elif MODALITY == "text":
            vectors = text_vectors
        elif MODALITY == "image":
            vectors = image_vectors
        else:
            vectors = id_vectors + text_vectors + torch.sigmoid(self.image_gate) * image_vectors
        return nn.functional.normalize(vectors, dim=1)

    def query(self, history, table=None):
        table = self.item_table() if table is None else table
        mask = history.ne(0)
        weights = torch.arange(1, history.shape[1] + 1, device=history.device)[None] * mask
        vectors = table[history]
        query = (vectors * weights[:, :, None]).sum(1) / weights.sum(1, keepdim=True).clamp_min(1)
        return nn.functional.normalize(query, dim=1)

## 4. Hàm huấn luyện

`fit()` dùng được cho cả hai lần: (a) train trên `train` + chọn epoch bằng cửa sổ selection, (b) refit trên train+selection+valid với số epoch cố định.

In [9]:
def degree_of(baskets):
    return np.bincount(np.asarray(flatten_baskets([b for days in baskets.values() for b in days]), dtype="int64"), minlength=N_ITEMS + 1).astype("float64")

def make_negative_prob(degree):
    freq = degree + 1.0
    freq[0] = 0  # PAD không được lấy làm negative.
    popular = freq ** 0.75
    popular /= popular.sum()
    uniform = np.zeros_like(popular)
    uniform[1:] = 1 / N_ITEMS
    ratio = float(os.getenv("HM_UNIFORM_NEG_RATIO", "0.0"))
    assert 0 <= ratio <= 1
    return torch.tensor(ratio * uniform + (1 - ratio) * popular, dtype=torch.float32, device=device)

def sample_negatives(neg_prob, target, history):
    """(B, N_NEG) negative theo popularity^0.75, loại positive hiện tại và lịch sử của chính user."""
    neg = torch.multinomial(neg_prob, target.numel() * N_NEG, replacement=True).view(-1, N_NEG)
    for _ in range(50):
        bad = (neg == target[:, None]) | (neg[:, :, None] == history[:, None, :]).any(2)
        if not bad.any():
            break
        neg[bad] = torch.multinomial(neg_prob, int(bad.sum()), replacement=True)
    return neg

def train_one_epoch(model, optimizer, loader, neg_prob):
    model.train()
    losses = []
    for history, target in loader:
        history, target = history.to(device), target.to(device)
        candidates = torch.cat([target[:, None], sample_negatives(neg_prob, target, history)], dim=1)

        table = model.item_table(id_drop=ID_DROPOUT)
        logits = (model.query(history, table)[:, None, :] * table[candidates]).sum(-1) / TEMPERATURE
        labels = torch.zeros(len(target), dtype=torch.long, device=device)
        loss = nn.functional.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))

## 5. Đánh giá

* **Full-ranking** (next-basket, như v1): xếp hạng toàn catalog, `HitRate/Recall/MAP/NDCG@K`, tách warm / strict-cold / repeat / explore.
* **Sampled 1+99** (như Amazon): với MỖI cặp (user, item mua trong cửa sổ đích), xếp hạng item đó cùng 99 negative lấy ngẫu nhiên từ catalog, loại lịch sử và mọi item user mua trong cửa sổ đích. Hai kiểu negative: `uniform` và `popularity` (khó hơn). Đồng hạng tính trung bình. CI 95% bootstrap theo user.
* Luôn in kèm `random` và `popularity`/`recent_popularity` cùng giao thức — **không so số sampled với số full-ranking**.

In [10]:
@torch.no_grad()
def score_matrix(model, context, users, table, serving=None):
    """(B, N_ITEMS+1) điểm tower cho các user; user không có lịch sử -> hàng toàn 0 (được xử lý ở chỗ gọi).
    serving = {"inactive": bool tensor | None, "bonus": float, "pop_term": tensor | None}: cộng w*log(1+bán 7 ngày), cộng điểm item cold
    (degree=0), loại item không bán trong `active_days` ngày."""
    histories = []
    for user in users:
        history = context.get(user, [])[-MAXLEN:]
        histories.append([0] * (MAXLEN - len(history)) + history)
    batch = torch.tensor(histories, device=device)
    scores = model.query(batch, table) @ table.T
    if serving:
        if serving["pop_term"] is not None:
            scores += serving["pop_term"][None]
        if serving["bonus"]:
            scores[:, model.degree == 0] += serving["bonus"]
        if serving["inactive"] is not None:
            scores[:, serving["inactive"]] = -torch.inf
    scores[:, 0] = -torch.inf
    return scores

def make_serving(window, active_days, bonus, pop_weight=0.0):
    if not active_days and not bonus and not pop_weight:
        return None
    inactive = None if not active_days else torch.tensor(window_counts(window, active_days) == 0, device=device)
    pop_term = None if not pop_weight else torch.tensor(pop_weight * np.log1p(window_counts(window, 7)), dtype=torch.float32, device=device)
    return {"inactive": inactive, "bonus": float(bonus), "pop_term": pop_term}

@torch.no_grad()
def rank_users(model, context, users, popular_ranked, topk=1000, batch_size=256, serving=None):
    """user -> top-k item idx (user không có lịch sử dùng popularity)."""
    model.eval()
    table = model.item_table()
    rankings, fallback = {}, 0
    for start in range(0, len(users), batch_size):
        chunk = users[start:start + batch_size]
        top = torch.topk(score_matrix(model, context, chunk, table, serving), topk, dim=1).indices.cpu().numpy()
        for user, row in zip(chunk, top):
            if context.get(user):
                rankings[user] = row.tolist()
            else:
                rankings[user] = popular_ranked[:topk]
                fallback += 1
    return rankings, fallback

def ranking_from_vector(vector, topk=1000):
    vector = vector.copy()
    vector[0] = -np.inf
    return [int(i) for i in np.argsort(-vector, kind="stable")[:topk]]

def full_ranking_metrics(rankings, targets, context=None, ks=(12, 50, 100, 500, 1000), target_filter=None, fallback=0):
    ks = tuple(sorted(set(ks)))
    top = max(ks)
    discounts = 1 / np.log2(np.arange(top) + 2)
    positions = np.arange(1, top + 1)
    values = {name: {k: [] for k in ks} for name in ("map", "recall", "hit", "ndcg")}
    recommended = set()
    for user, truth in targets.items():
        relevant = set(truth)
        if target_filter is not None:
            relevant = {item for item in relevant if target_filter(user, item)}
        if not relevant or user not in rankings:
            continue
        ranked = np.asarray(rankings[user][:top])
        recommended.update(ranked.tolist())
        hits = np.isin(ranked, list(relevant)).astype("float64")
        precision = np.cumsum(hits) / positions[:len(hits)]
        for k in ks:
            n_rel = len(relevant)
            values["map"][k].append(float((precision[:k] * hits[:k]).sum()) / min(k, n_rel))
            values["recall"][k].append(float(hits[:k].sum()) / n_rel)
            values["hit"][k].append(float(hits[:k].any()))
            values["ndcg"][k].append(float((hits[:k] * discounts[:len(hits)][:k]).sum()) / float(discounts[:min(k, n_rel)].sum()))
    n_users = len(values["map"][ks[0]])
    result = {"evaluated_users": n_users, "fallback_popularity_users": fallback, f"CatalogCoverage@{top}": len(recommended) / N_ITEMS}
    for k in ks:
        for key, name in (("map", "MAP"), ("recall", "Recall"), ("hit", "HitRate"), ("ndcg", "NDCG")):
            result[f"{name}@{k}"] = float(np.mean(values[key][k]) if values[key][k] else 0)
        result[f"RecallLiftVsRandom@{k}"] = result[f"Recall@{k}"] / (k / N_ITEMS)
    return result

def selection_recall(model, popular_ranked, serving=None):
    """Recall@100 / HitRate@100 trên cửa sổ chọn checkpoint (context = train)."""
    users = [u for u in selection_window if selection_window[u] and u in train]
    rankings, _ = rank_users(model, train, users, popular_ranked, topk=100, serving=serving)
    return full_ranking_metrics(rankings, {u: selection_window[u] for u in users}, ks=(12, 50, 100))

In [11]:
def draw_negatives(rng, n_rows, n_neg, forbidden, cdf=None):
    out = np.empty((n_rows, n_neg), dtype=np.int64)
    forbidden = np.fromiter(forbidden, dtype=np.int64)
    for row in range(n_rows):
        got = []
        while len(got) < n_neg:
            need = n_neg - len(got)
            if cdf is None:
                draws = rng.integers(1, N_ITEMS + 1, size=need * 2)
            else:
                draws = np.clip(np.searchsorted(cdf, rng.random(need * 2)) + 1, 1, N_ITEMS)
            draws = draws[~np.isin(draws, forbidden)]
            got = list(dict.fromkeys(got + draws.tolist()))[:n_neg]
        out[row] = got
    return out

def cluster_ci(hits_by_pair, user_index, n_users, z=1.96):
    """CI của tỉ lệ (tổng hit / tổng cặp) với cluster = user (ratio estimator)."""
    hits_by_pair = np.asarray(hits_by_pair, dtype="float64")
    n = len(hits_by_pair)
    if n == 0:
        return float("nan")
    mean = hits_by_pair.mean()
    sums = np.bincount(user_index, weights=hits_by_pair - mean, minlength=n_users)
    return float(z * np.sqrt((sums ** 2).sum()) / n)

@torch.no_grad()
def sampled_eval(score_fn, context, targets, strategy="uniform", n_neg=SAMPLED_NEG, ks=(10, 20, 50),
                 neg_weights=None, target_filter=None, seed=SEED, batch_size=128):
    """1 positive + n_neg negative cho từng cặp (user, item đích). score_fn(users) -> (B, N_ITEMS+1) tensor."""
    rng = np.random.default_rng(seed)
    cdf = None
    if strategy == "popularity":
        w = np.maximum(neg_weights[1:].astype("float64"), 1e-6)
        cdf = np.cumsum(w / w.sum()); cdf[-1] = 1.0
    users = [u for u, truth in targets.items() if truth and u in context]
    ranks, user_idx, positive_kind = [], [], []
    for start in range(0, len(users), batch_size):
        chunk = users[start:start + batch_size]
        scores = score_fn(chunk)
        rows, cands = [], []
        for i, user in enumerate(chunk):
            truth = sorted(set(targets[user]))
            positives = [t for t in truth if target_filter is None or target_filter(user, t)]
            if not positives:
                continue
            forbidden = set(context.get(user, [])) | set(truth) | {0}
            negs = draw_negatives(rng, len(positives), n_neg, forbidden, cdf)
            cands.append(np.concatenate([np.asarray(positives)[:, None], negs], axis=1))
            rows.extend([i] * len(positives))
            user_idx.extend([start + i] * len(positives))
        if not rows:
            continue
        cands = np.concatenate(cands)
        vals = scores[torch.tensor(rows, device=scores.device)[:, None], torch.from_numpy(cands).to(scores.device)].cpu().numpy()
        pos, neg = vals[:, :1], vals[:, 1:]
        ranks.append((neg > pos).sum(1) + 0.5 * (neg == pos).sum(1))
    if not ranks:
        return {"n_pairs": 0}
    ranks = np.concatenate(ranks)
    user_idx = np.asarray(user_idx)
    result = {"n_pairs": int(len(ranks)), "n_users": int(len(set(user_idx.tolist()))), "pool": 1 + n_neg, "strategy": strategy, "MRR": float((1 / (ranks + 1)).mean())}
    for k in ks:
        hit = (ranks < k).astype("float64")
        result[f"HitRate@{k}"] = float(hit.mean())
        result[f"HitRate@{k}_ci95"] = cluster_ci(hit, user_idx, len(users))
        result[f"NDCG@{k}"] = float(np.where(ranks < k, 1 / np.log2(ranks + 2), 0).mean())
    return result

def tower_score_fn(model, context):
    model.eval()
    table = model.item_table()
    return lambda users: score_matrix(model, context, users, table)

def vector_score_fn(vector):
    vec = torch.tensor(vector, dtype=torch.float32, device=device)
    return lambda users: vec[None].expand(len(users), -1)

def random_score_fn(seed=SEED):
    gen = torch.Generator(device=device).manual_seed(seed)
    return lambda users: torch.rand(len(users), N_ITEMS + 1, device=device, generator=gen)

# Kiểm tra giao thức: oracle phải đạt 100%, random phải ~ k/pool.
_users = list(valid)[:200]
_oracle = lambda users: torch.stack([torch.isin(torch.arange(N_ITEMS + 1, device=device), torch.tensor(valid[u], device=device)).float() for u in users])
_o = sampled_eval(_oracle, {u: train.get(u, []) for u in _users}, {u: valid[u] for u in _users})
_r = sampled_eval(random_score_fn(), {u: train.get(u, []) for u in _users}, {u: valid[u] for u in _users})
assert _o["HitRate@10"] > 0.99, _o
print(f"sanity 1+{SAMPLED_NEG}: oracle HR@10={_o['HitRate@10']:.3f} | random HR@10={_r['HitRate@10']:.3f} (kỳ vọng {10 / (1 + SAMPLED_NEG):.3f})")

sanity 1+99: oracle HR@10=1.000 | random HR@10=0.108 (kỳ vọng 0.100)


## 6. Huấn luyện và chọn checkpoint

Dùng cửa sổ chọn checkpoint (đọc từ file `rerank_train.parquet`) để chọn epoch tốt nhất theo Recall@100, **đã áp cold-fix** nếu bật (để chọn đúng cấu hình sẽ triển khai). Validation và test chưa được dùng ở bước này.

In [12]:
def fit(fit_baskets, epochs, select=False, tag="base"):
    debug_users = int(os.getenv("HM_DEBUG_USERS", 0))  # chỉ để smoke-test nhanh
    if debug_users:
        fit_baskets = dict(list(fit_baskets.items())[:debug_users])
    degree = degree_of(fit_baskets)
    popular_ranked = [int(i) for i in np.argsort(-degree, kind="stable") if i != 0]
    neg_prob = make_negative_prob(degree)
    dataset = PairData(fit_baskets)
    workers = int(os.getenv("HM_NUM_WORKERS", 2 if os.name != "nt" else 0))
    loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=workers, pin_memory=device == "cuda")
    torch.manual_seed(SEED)
    model = Tower().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    print(f"[{tag}] pairs: {len(dataset):,} | catalog: {N_ITEMS:,} | items with degree>0: {int((degree > 0).sum()):,} | epochs: {epochs}")

    history, best, best_state, best_epoch, patience, bad = [], -1.0, None, epochs, 3, 0
    for epoch in range(1, epochs + 1):
        started = time.time()
        row = {"epoch": epoch, "loss": train_one_epoch(model, optimizer, loader, neg_prob)}
        if select:
            model.set_degree(degree, 1 if COLD_FIX else 0)
            row.update(selection_recall(model, popular_ranked))
        row["seconds"] = round(time.time() - started, 1)
        history.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items() if k in ("epoch", "loss", "Recall@100", "HitRate@100", "MAP@12", "seconds")})
        if select:
            if row["Recall@100"] > best:
                best, best_epoch, bad = row["Recall@100"], epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                print(f"Patience counter: {bad}/{patience}")
                if bad >= patience:
                    print(f"Early stopping at epoch {epoch}.")
                    break
    if select:
        model.load_state_dict(best_state)
    model.set_degree(degree, 0)
    return model, degree, popular_ranked, history, best_epoch

epochs = int(os.getenv("HM_EPOCHS", 8 if PROFILE == "quick" else 20))
model_base, degree_base, popular_base, history, best_epoch = fit(train_baskets, epochs, select=True, tag="base")
(OUTPUT_DIR / "retrieval_history.json").write_text(json.dumps(history, indent=2))
print("best epoch:", best_epoch)

[base] pairs: 985,376 | catalog: 105,542 | items with degree>0: 71,357 | epochs: 8
{'epoch': 1, 'loss': 4.6793, 'MAP@12': 0.0067, 'Recall@100': 0.0494, 'HitRate@100': 0.1002, 'seconds': 75.4}
{'epoch': 2, 'loss': 4.2775, 'MAP@12': 0.0082, 'Recall@100': 0.0569, 'HitRate@100': 0.1133, 'seconds': 75.3}
{'epoch': 3, 'loss': 4.2177, 'MAP@12': 0.008, 'Recall@100': 0.0584, 'HitRate@100': 0.1159, 'seconds': 75.6}
{'epoch': 4, 'loss': 4.1654, 'MAP@12': 0.0082, 'Recall@100': 0.0609, 'HitRate@100': 0.1237, 'seconds': 75.6}
{'epoch': 5, 'loss': 4.1119, 'MAP@12': 0.0087, 'Recall@100': 0.0613, 'HitRate@100': 0.1254, 'seconds': 75.6}
{'epoch': 6, 'loss': 4.0615, 'MAP@12': 0.0087, 'Recall@100': 0.0602, 'HitRate@100': 0.1244, 'seconds': 75.6}
Patience counter: 1/3
{'epoch': 7, 'loss': 4.0094, 'MAP@12': 0.0101, 'Recall@100': 0.068, 'HitRate@100': 0.1312, 'seconds': 75.5}
{'epoch': 8, 'loss': 3.9563, 'MAP@12': 0.0108, 'Recall@100': 0.0625, 'HitRate@100': 0.1247, 'seconds': 75.6}
Patience counter: 1/3
bes

### Chọn ngưỡng cold-fix

Quét ngưỡng degree trên cửa sổ chọn checkpoint (không dùng valid/test). `0` = không tắt ID nào (v1).

In [13]:
def tune_cold_threshold(model, degree, popular_ranked):
    rows = {}
    for threshold in [0] + COLD_GRID:
        model.set_degree(degree, threshold)
        m = selection_recall(model, popular_ranked)
        rows[threshold] = {k: m[k] for k in ("Recall@12", "Recall@50", "Recall@100", "HitRate@100")}
        print(f"min_degree={threshold:>2}: " + " | ".join(f"{k}={v:.4f}" for k, v in rows[threshold].items()))
    if not COLD_FIX:
        return 0, rows
    best = max(rows, key=lambda t: rows[t]["Recall@100"])
    return best, rows

cold_threshold, cold_grid_rows = tune_cold_threshold(model_base, degree_base, popular_base)
model_base.set_degree(degree_base, cold_threshold)
print("chọn min_degree =", cold_threshold)

min_degree= 0: Recall@12=0.0245 | Recall@50=0.0536 | Recall@100=0.0744 | HitRate@100=0.1450
min_degree= 1: Recall@12=0.0216 | Recall@50=0.0443 | Recall@100=0.0680 | HitRate@100=0.1312
min_degree= 2: Recall@12=0.0212 | Recall@50=0.0417 | Recall@100=0.0653 | HitRate@100=0.1263
min_degree= 3: Recall@12=0.0225 | Recall@50=0.0444 | Recall@100=0.0670 | HitRate@100=0.1267
min_degree= 5: Recall@12=0.0235 | Recall@50=0.0430 | Recall@100=0.0645 | HitRate@100=0.1234
min_degree=10: Recall@12=0.0213 | Recall@50=0.0422 | Recall@100=0.0626 | HitRate@100=0.1231
chọn min_degree = 0


### Luật phục vụ (suy luận): item đang bán + độ bán chạy + cộng điểm item cold

Mỗi tuần chỉ ~4,5k/105k item được bán, nhưng tower xếp hạng cả catalog và không có khái niệm "đang bán". Ba chỉnh sửa **chỉ ở lúc suy luận** (không đổi trọng số): điểm = điểm tower + `pop_weight`·log(1 + số lần bán 7 ngày trước cutoff) + `cold_bonus` (nếu item cold, degree=0); loại item không có giao dịch trong `active_days` ngày. 3 tham số này được quét bằng Recall@100 trên cửa sổ **valid** (model_base chưa thấy valid): cửa sổ selection không dùng được vì theo định nghĩa mọi item bán trước nó đều đã có trong train, nên `cold_bonus` không có tác dụng ở đó. Hệ quả: số **valid** của tower không còn là số sạch cho 3 tham số này; số **test** vẫn sạch. Bảng sampled 1+99 vẫn chấm tower thô.

In [14]:
validation_context = {user: train.get(user, []) + selection_window.get(user, []) for user in valid}

def tune_serving(model, popular_ranked):
    grid = {}
    if not TUNE_SERVING:
        return (0, 0.0, 0.0), grid
    users = [u for u in valid if validation_context.get(u)]
    targets = {u: valid[u] for u in users}
    for days in ACTIVE_GRID:
        for bonus in BONUS_GRID:
            for pop_w in POP_GRID:
                rankings, _ = rank_users(model, validation_context, users, popular_ranked, topk=100, serving=make_serving("valid", days, bonus, pop_w))
                m = full_ranking_metrics(rankings, targets, ks=(12, 50, 100))
                grid[(days, bonus, pop_w)] = {k: m[k] for k in ("HitRate@12", "Recall@100", "HitRate@100")}
    best = max(grid, key=lambda key: (grid[key]["Recall@100"], -key[1], -key[2]))
    top5 = sorted(grid, key=lambda key: -grid[key]["Recall@100"])[:5]
    for key in top5:
        print(f"active_days={key[0]:>2} cold_bonus={key[1]:<5} pop_weight={key[2]:<5}: " + " | ".join(f"{k}={v:.4f}" for k, v in grid[key].items()))
    print(f"không luật (0,0,0): " + " | ".join(f"{k}={v:.4f}" for k, v in grid[(0, 0.0, 0.0)].items()))
    return best, grid

(serving_days, serving_bonus, serving_pop), serving_grid = tune_serving(model_base, popular_base)
print(f"chọn active_days={serving_days}, cold_bonus={serving_bonus}, pop_weight={serving_pop}")
serving_cfg = {"active_days": serving_days, "cold_bonus": serving_bonus, "pop_weight": serving_pop}
torch.save({"state": model_base.state_dict(), "item_ids": item_ids, "d_model": D_MODEL, "degree": degree_base,
            "cold_min_degree": cold_threshold, "best_epoch": best_epoch, "valid": cold_grid_rows[cold_threshold], "modality": MODALITY,
            "id_dropout": ID_DROPOUT, "serving": serving_cfg}, OUTPUT_DIR / "best_retrieval.pt")

active_days= 0 cold_bonus=0.2   pop_weight=0.1  : HitRate@12=0.0960 | Recall@100=0.1550 | HitRate@100=0.3109
active_days=14 cold_bonus=0.2   pop_weight=0.1  : HitRate@12=0.0960 | Recall@100=0.1550 | HitRate@100=0.3109
active_days=28 cold_bonus=0.2   pop_weight=0.1  : HitRate@12=0.0960 | Recall@100=0.1550 | HitRate@100=0.3109
active_days= 0 cold_bonus=0.2   pop_weight=0.05 : HitRate@12=0.1050 | Recall@100=0.1544 | HitRate@100=0.3073
active_days=28 cold_bonus=0.2   pop_weight=0.05 : HitRate@12=0.1050 | Recall@100=0.1543 | HitRate@100=0.3073
không luật (0,0,0): HitRate@12=0.0455 | Recall@100=0.0650 | HitRate@100=0.1343
chọn active_days=0, cold_bonus=0.2, pop_weight=0.1


### Refit cho test

Train lại từ đầu trên **train + selection + valid** với đúng `best_epoch` epoch và cùng ngưỡng cold-fix. Item mới xuất hiện trong 2 tuần cuối giờ có tín hiệu tương tác. Valid vẫn được chấm bằng `model_base` (chưa thấy valid).

In [15]:
if REFIT:
    refit_baskets = merge_baskets(train_baskets, selection_baskets, valid_baskets)
    model_refit, degree_refit, popular_refit, _, _ = fit(refit_baskets, best_epoch, select=False, tag="refit")
    model_refit.set_degree(degree_refit, cold_threshold)
    torch.save({"state": model_refit.state_dict(), "item_ids": item_ids, "d_model": D_MODEL, "degree": degree_refit,
                "cold_min_degree": cold_threshold, "best_epoch": best_epoch, "modality": MODALITY, "id_dropout": ID_DROPOUT, "serving": serving_cfg}, OUTPUT_DIR / "best_retrieval_refit.pt")
else:
    model_refit, degree_refit, popular_refit = None, None, None

[refit] pairs: 1,003,726 | catalog: 105,542 | items with degree>0: 72,529 | epochs: 7
{'epoch': 1, 'loss': 4.6737, 'seconds': 77.9}
{'epoch': 2, 'loss': 4.2774, 'seconds': 77.2}
{'epoch': 3, 'loss': 4.2175, 'seconds': 77.1}
{'epoch': 4, 'loss': 4.1647, 'seconds': 77.3}
{'epoch': 5, 'loss': 4.1115, 'seconds': 77.6}
{'epoch': 6, 'loss': 4.0606, 'seconds': 77.0}
{'epoch': 7, 'loss': 4.009, 'seconds': 77.8}


## 7. Đánh giá trên validation và test

Context cho valid = train + selection; context cho test = train + selection + valid. Với test, in cả `model_base` lẫn `model_refit` để thấy tác động của refit.

In [16]:
def evaluate_slices(model, degree, popular_ranked, context, targets, window):
    """Full-ranking (nhiều lát) + sampled 1+99, kèm baseline random/popularity/recent_popularity."""
    users = list(targets)
    serving = make_serving(window, serving_cfg["active_days"], serving_cfg["cold_bonus"], serving_cfg["pop_weight"])
    rankings, fallback = rank_users(model, context, users, popular_ranked, serving=serving)
    raw_rankings, _ = rank_users(model, context, users, popular_ranked)
    warm = lambda _, item: degree[item] > 0
    cold = lambda _, item: degree[item] == 0
    repeat = lambda user, item: item in context.get(user, [])
    explore = lambda user, item: item not in context.get(user, [])
    report = {"full_ranking": {
        "overall": full_ranking_metrics(rankings, targets, fallback=fallback),
        "warm_items": full_ranking_metrics(rankings, targets, target_filter=warm),
        "strict_cold_items": full_ranking_metrics(rankings, targets, target_filter=cold),
        "repeat_items": full_ranking_metrics(rankings, targets, target_filter=repeat),
        "explore_items": full_ranking_metrics(rankings, targets, target_filter=explore),
    }}
    pop_rank = {u: popular_ranked[:1000] for u in users}
    recent_top = ranking_from_vector(recent_pop[window])
    recent_rank = {u: recent_top for u in users}
    report["full_ranking_baselines"] = {
        "popularity": full_ranking_metrics(pop_rank, targets),
        "recent_popularity": full_ranking_metrics(recent_rank, targets),
        "tower_without_serving_rule": full_ranking_metrics(raw_rankings, targets),
    }
    cold_share = float(np.mean([degree[i] == 0 for u in users for i in targets[u]]))
    report["cold_target_share"] = cold_share

    scorers = {"tower": tower_score_fn(model, context), "popularity": vector_score_fn(degree), "recent_popularity": vector_score_fn(recent_pop[window]), "random": random_score_fn()}
    sampled = {}
    for strategy in ("uniform", "popularity"):
        for slice_name, flt in (("overall", None), ("warm_items", warm), ("strict_cold_items", cold)):
            sampled[f"{strategy}/{slice_name}"] = {
                name: sampled_eval(fn, context, targets, strategy=strategy, neg_weights=degree, target_filter=flt)
                for name, fn in scorers.items() if flt is None or name in ("tower", "random")}
    report["sampled_1plus99"] = sampled
    return report

def print_summary(title, report):
    fr = report["full_ranking"]
    print(f"\n===== {title} | cold target share {report['cold_target_share']:.1%} =====")
    print(f"{'full-ranking':<26}{'Hit@12':>8}{'Hit@50':>8}{'Hit@100':>9}{'Rec@100':>9}{'NDCG@12':>9}")
    rows = list(fr.items()) + [(f"[base] {k}", v) for k, v in report["full_ranking_baselines"].items()]
    for name, m in rows:
        print(f"{name:<26}{m['HitRate@12']:>8.4f}{m['HitRate@50']:>8.4f}{m['HitRate@100']:>9.4f}{m['Recall@100']:>9.4f}{m['NDCG@12']:>9.4f}  (users={m['evaluated_users']})")
    print(f"\n{'sampled 1+%d' % SAMPLED_NEG:<34}{'HR@10':>8}{'±ci':>7}{'HR@20':>8}{'NDCG@10':>9}{'MRR':>7}{'pairs':>7}")
    for key, by_model in report["sampled_1plus99"].items():
        for name, m in by_model.items():
            if m.get("n_pairs"):
                print(f"{key + ' / ' + name:<34}{m['HitRate@10']:>8.4f}{m['HitRate@10_ci95']:>7.3f}{m['HitRate@20']:>8.4f}{m['NDCG@10']:>9.4f}{m['MRR']:>7.3f}{m['n_pairs']:>7}")

test_context = {user: train.get(user, []) + selection_window.get(user, []) + valid.get(user, []) for user in test}

result = {"modality": MODALITY, "selection_split": "retrieval_selection_window", "selection_source_file": "rerank_train.parquet",
          "config": {"id_dropout": ID_DROPOUT, "cold_fix": COLD_FIX, "cold_min_degree": cold_threshold, "refit": REFIT, "best_epoch": best_epoch, "n_neg_train": N_NEG, "sampled_neg": SAMPLED_NEG},
          "cold_threshold_grid_selection_window": cold_grid_rows, "serving_rule": serving_cfg, "serving_grid_valid_window": {f"{k[0]}d/bonus{k[1]}/pop{k[2]}": v for k, v in serving_grid.items()}, "stats_source": STATS_SOURCE,
          "valid": evaluate_slices(model_base, degree_base, popular_base, validation_context, valid, "valid"),
          "test_base_model": evaluate_slices(model_base, degree_base, popular_base, test_context, test, "test")}
if REFIT:
    result["test"] = evaluate_slices(model_refit, degree_refit, popular_refit, test_context, test, "test")
else:
    result["test"] = result["test_base_model"]

print_summary("VALID (model_base)", result["valid"])
print_summary("TEST (model_base, chưa thấy 2 tuần cuối)", result["test_base_model"])
if REFIT:
    print_summary("TEST (model_refit) <- số báo cáo", result["test"])

(OUTPUT_DIR / "retrieval_metrics.json").write_text(json.dumps(result, indent=2))


===== VALID (model_base) | cold target share 23.1% =====
full-ranking                Hit@12  Hit@50  Hit@100  Rec@100  NDCG@12
overall                     0.0960  0.2234   0.3109   0.1550   0.0274  (users=3030)
warm_items                  0.1024  0.2338   0.3229   0.1847   0.0323  (users=2784)
strict_cold_items           0.0066  0.0323   0.0587   0.0397   0.0017  (users=1209)
repeat_items                0.2344  0.3769   0.4866   0.4614   0.1297  (users=337)
explore_items               0.0762  0.1976   0.2850   0.1371   0.0190  (users=2940)
[base] popularity           0.0284  0.0776   0.1083   0.0412   0.0071  (users=3030)
[base] recent_popularity    0.0548  0.1498   0.2281   0.1001   0.0112  (users=3030)
[base] tower_without_serving_rule  0.0455  0.0983   0.1343   0.0650   0.0142  (users=3030)

sampled 1+99                         HR@10    ±ci   HR@20  NDCG@10    MRR  pairs
uniform/overall / tower             0.5388  0.013  0.6828   0.3314  0.286   9532
uniform/overall / popularity   

74741

**Ghi chú:**
* Checkpoint được chọn theo Recall@100 trên cửa sổ chọn checkpoint (file `rerank_train.parquet`); số epoch và ngưỡng cold-fix cũng chỉ lấy từ cửa sổ này. Validation/test chỉ dùng ở bước đánh giá cuối.
* Số **headline** là full-ranking của `test` (model_refit). Bảng sampled 1+99 chỉ là bảng phụ để so với Amazon/literature: cùng giao thức mới so sánh được, và luôn đọc cạnh `random`/`popularity`.
* `best_retrieval.pt` (base, đã có `degree`, `cold_min_degree`) và `best_retrieval_refit.pt` được notebook 03 đọc.